# 4.1 — Direct Muscle Control: Hand and Fingers

This notebook drives individual finger muscles directly (bypassing RL) to show how MyoSuite's hand model responds to raw muscle excitation commands.

**What you'll learn:**
- How to disable action normalization and send raw muscle excitations
- Which MuJoCo actuator names correspond to finger flexors/extensors
  - `FDP2_r` = Flexor Digitorum Profundus (index finger flexor, right hand)
  - `EDC2_r` = Extensor Digitorum Communis (index finger extensor, right hand)

**Prerequisites:** Completed notebook 1.1 · `myosuite` installed

In [ ]:
from myosuite.utils import gym
import numpy as np
import os
from myosuite.utils.video_io import show_video
from myosuite.utils.video_io import write_video


In [ ]:
import mujoco
env = gym.make('myoHandPoseRandom-v0', normalize_act=False, render_mode='rgb_array')

env.unwrapped.init_qpos[:] = np.zeros(len(env.unwrapped.init_qpos),)
mjcModel = env.unwrapped.model

# Right-hand actuators use a `_r` suffix (e.g. FDP2_r, EDC2_r).
musc_fe = [mjcModel.actuator('FDP2_r').id, mjcModel.actuator('EDC2_r').id]
L_range = round(1 / mjcModel.opt.timestep)
skip_frame = 50
env.reset()

frames_sim = []
for iter_n in range(3):
    print("iteration: " + str(iter_n))
    for rp in range(2):  # alternate between flexor and extensor
        for s in range(L_range):
            if not (s % skip_frame):
                frames_sim.append(env.render())

            ctrl = np.zeros(mjcModel.na,)
            act_val = 1  # maximum muscle activation
            if rp == 0:
                ctrl[musc_fe[0]] = act_val
                ctrl[musc_fe[1]] = 0
            else:
                ctrl[musc_fe[1]] = act_val
                ctrl[musc_fe[0]] = 0
            env.step(ctrl)

os.makedirs('videos', exist_ok=True)
# make a local copy
write_video('videos/MyoSuite.mp4', np.asarray(frames_sim), outputdict={"-pix_fmt": "yuv420p"})

# show in the notebook
show_video('videos/MyoSuite.mp4')